# C08_E01 — Cinco lazos canónicos comparados

**Capítulo:** 8 — Dinámica de procesos industriales  
**Identificador:** `C08_E01`  
**Objetivo pedagógico:** Reconocer las firmas dinámicas típicas por familia de variable de proceso.  
**Librerías:** numpy, scipy.integrate, matplotlib

## Ejemplo industrial

P&ID con cinco lazos canónicos: caudal, presión, temperatura, nivel, composición.

---

> **Aviso de uso responsable.** Este notebook es didáctico. No es código de producción. Cualquier implementación real requiere validación independiente. Ver `docs/politica_uso_responsable.md`.

## 1. Modelos canónicos por familia

In [1]:
import numpy as np, matplotlib.pyplot as plt
from scipy.integrate import solve_ivp
import os
np.random.seed(0)

def primer_orden(t, y, K, tau):  return [(K - y[0])/tau]
def integrador(t, y, K):         return [K]

defs = {
    "Caudal":       ('FO', 1.0, 2.0,    'C0'),     # tau pequeno
    "Presión gas":  ('FO', 1.0, 5.0,    'C1'),
    "Temperatura":  ('FO', 1.0, 60.0,   'C2'),
    "Nivel":        ('INT', 0.05, None, 'C3'),
    "Composición":  ('FO', 1.0, 600.0,  'C4'),
}

## 2. Simulación y figura comparativa

In [2]:
fig, ax = plt.subplots(figsize=(8, 4.5))
t_full = np.linspace(0, 1500, 800)
for name, (kind, K, tau, color) in defs.items():
    if kind == 'FO':
        sol = solve_ivp(primer_orden, (0, 1500), [0.0], t_eval=t_full, args=(K, tau))
    else:
        sol = solve_ivp(integrador, (0, 1500), [0.0], t_eval=t_full, args=(K,))
    y = sol.y[0]
    yn = y / (np.max(np.abs(y)) + 1e-9)
    ax.plot(sol.t, yn, label=name, color=color)
ax.set_xlabel("t (s)"); ax.set_ylabel("y (normalizada)")
ax.legend(); ax.grid(alpha=0.3)
ax.set_title("Cinco lazos canónicos — respuestas comparadas")
out = '../../figures/cap08/fig_C08_F01_cinco_lazos.png'
os.makedirs(os.path.dirname(out), exist_ok=True)
fig.tight_layout(); fig.savefig(out, dpi=120); plt.show()

## 3. Interpretación

Las cinco firmas dinámicas son claramente distintas: caudal y presión-gas se estabilizan en segundos; temperatura en minutos; composición no se ha estabilizado aún a los 1500 s; nivel crece linealmente (integrador). La sintonía debe respetar esta jerarquía: aplicar la sintonía de un lazo de caudal a uno de composición es un error categórico.